In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.show_dimensions", True)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

Tasks:
• Handle missing values (decide whether to drop, impute, or flag).

• Convert categorical fields to numeric (encoding).

• Normalize numerical features if needed.

• Create train/test split: use the most recent month of available data as the test set, 
and the X months immediately preceding it as the training set. X is not fixed —
treat the training window length as a tunable choice and experiment to determine 
the optimal value of X.

• Deliverable: 02_preprocessing.ipynb + cleaned CSV.

In [2]:
"""
Load and combine CRMLS sold data from May 2025 to May 2026.
Restrict the dataset to California residential single-family properties.
"""

path = "../data/all_sales_data/"
target_months = [f"2025{m:02d}" for m in range(5, 13)] + [f"2026{m:02d}" for m in range(1, 6)]

all_files = []

for month in target_months:
    file_name = f"CRMLSSold{month}.csv"
    file_path = os.path.join(path, file_name)
    
    if os.path.exists(file_path):
        all_files.append(file_path)
    else:
        print(f"File not found: {file_path}")

if not all_files:
    raise FileNotFoundError("No files found for the specified months.")

df_raw = pd.concat(
    (
        pd.read_csv(
            f,
            low_memory=False,
            dtype={"PostalCode": "string"}
        )
        for f in all_files
    ),
    ignore_index=True
)

df_raw.columns = df_raw.columns.str.strip()

df_raw["ClosePrice"] = pd.to_numeric(df_raw["ClosePrice"], errors="coerce")
df_raw["CloseDate"] = pd.to_datetime(df_raw["CloseDate"], errors="coerce")

print("Raw shape:", df_raw.shape)

Raw shape: (281823, 78)


### Data Cleaning

In [3]:
"""
Row filtering:
- Keep California properties only
- Keep Residential + SingleFamilyResidence only
- Remove missing or non-positive target values
"""

df = df_raw[
    (df_raw["StateOrProvince"] == "CA") &
    (df_raw["PropertyType"] == "Residential") &
    (df_raw["PropertySubType"] == "SingleFamilyResidence") &
    (df_raw["ClosePrice"].notna()) &
    (df_raw["ClosePrice"] > 0)
].copy()

print("After row filtering:", df.shape)

After row filtering: (141990, 78)


In [4]:
"""
Observation:
"""
# LotSizeSquareFeet and LotSizeAcres are perfectly correlated, so only one should be kept.
# LotSizeArea is not strongly correlated with the other lot size fields and its unit is unclear.
# For the baseline model, I keep LotSizeSquareFeet and drop LotSizeAcres and LotSizeArea.

lot_cols = ["LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea"]
lot_df = df[lot_cols].apply(pd.to_numeric, errors="coerce")

print("Missing rate:")
print(lot_df.isnull().mean())

print("\nCorrelation:")
display(lot_df.corr())

print("--------------")
#I removed Stories and kept Levels because the two variables are highly redundant, 
# but Levels has fewer missing values and captures more detailed structural information.
# While Stories mainly contains 1, 2, or missing values, Levels includes additional categories such as "ThreeOrMore" and "MultiSplit". 
# Therefore, Levels is more informative for the baseline model.
print(df[["Stories", "Levels"]].isnull().mean())
print(df["Stories"].value_counts(dropna=False))
print(df["Levels"].value_counts(dropna=False))
pd.crosstab(df["Stories"], df["Levels"], dropna=False)


Missing rate:
LotSizeSquareFeet    0.017135
LotSizeAcres         0.017142
LotSizeArea          0.017043
Length: 3, dtype: float64

Correlation:


,LotSizeSquareFeet,LotSizeAcres,LotSizeArea
LotSizeSquareFeet,1.000000,1.000000,0.011685
LotSizeAcres,1.000000,1.000000,0.008207
LotSizeArea,0.011685,0.008207,1.000000


--------------
Stories    0.104043
Levels     0.071935
Length: 2, dtype: float64
Stories
1.0    82413
2.0    44804
NaN    14773
Name: count, Length: 3, dtype: int64
Levels
One                               81138
Two                               44794
NaN                               10214
ThreeOrMore                        2737
MultiSplit                         2067
One,Two                             292
Two,One                             264
Two,MultiSplit                      210
ThreeOrMore,MultiSplit               88
One,MultiSplit                       73
Two,ThreeOrMore                      45
MultiSplit,One                       26
One,ThreeOrMore                      17
One,Two,MultiSplit                    7
ThreeOrMore,One                       6
One,Two,ThreeOrMore                   4
Two,ThreeOrMore,MultiSplit            4
One,Two,ThreeOrMore,MultiSplit        2
Two,MultiSplit,One                    2
Name: count, Length: 19, dtype: int64


Levels,MultiSplit,"MultiSplit,One",One,"One,MultiSplit","One,ThreeOrMore","One,Two","One,Two,MultiSplit","One,Two,ThreeOrMore","One,Two,ThreeOrMore,MultiSplit",ThreeOrMore,"ThreeOrMore,MultiSplit","ThreeOrMore,One",Two,"Two,MultiSplit","Two,MultiSplit,One","Two,One","Two,ThreeOrMore","Two,ThreeOrMore,MultiSplit",NaN
Stories,,,,,,,,,,,,,,,,,,,
1.0,14,26,81138,73,17,292,7,4,2,2,0,6,248,1,2,264,0,0,317
2.0,0,0,0,0,0,0,0,0,0,0,0,0,44546,209,0,0,45,4,0
NaN,2053,0,0,0,0,0,0,0,0,2735,88,0,0,0,0,0,0,0,9897


In [5]:
'''
Feature Selection
'''
#high missing_cols as columns with more than 60% missing values
missing_pct = df.isnull().mean()
high_missing_cols = missing_pct[missing_pct > 0.6]
print(high_missing_cols.sort_values(ascending=False))

#

drop_cols = set([
    # filter / status columns
    "PropertyType",
    "PropertySubType",
    "MlsStatus",
    "StateOrProvince", #CA only

    # identifiers
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    "BuyerAgentMlsId",

    # agent / office / buyer fields
    "BuyerAgentAOR",
    "ListAgentAOR",
    "ListAgentEmail",
    "ListAgentFirstName",
    "ListAgentLastName",
    "ListAgentFullName",
    "ListOfficeName",
    "BuyerOfficeName",
    "BuyerOfficeAOR",
    "BuyerAgentFirstName",
    "BuyerAgentLastName",
    "CoBuyerAgentFirstName",
    "CoListAgentFirstName",
    "CoListAgentLastName",
    "CoListOfficeName",

    # leakage / listing-process columns
    "ListPrice",
    "OriginalListPrice",
    "DaysOnMarket",
    "ContractStatusChangeDate",
    "PurchaseContractDate",
    "ListingContractDate",

    # exact address / noisy granular location
    "UnparsedAddress",
    "StreetNumberNumeric",

    #correlated columns
    "LotSizeAcres",
    "LotSizeArea",
    "Stories",

    #Coarse-grained regional data
    'CountyOrParish',
    'MLSAreaMajor', ## By observing, it does not contain much information.
    "City"
])

drop_cols.update(high_missing_cols.index.tolist())

# Keep the target variable
drop_cols.discard("ClosePrice")

df_current = df.drop(columns=list(drop_cols), errors="ignore")

print("After feature selection:", df_current.shape)
print(df_current.columns)


FireplacesTotal                 1.000000
MiddleOrJuniorSchoolDistrict    1.000000
BusinessType                    1.000000
TaxYear                         1.000000
ElementarySchoolDistrict        1.000000
TaxAnnualAmount                 1.000000
AboveGradeFinishedArea          1.000000
CoveredSpaces                   1.000000
WaterfrontYN                    0.999486
BelowGradeFinishedArea          0.992816
BasementYN                      0.975822
BuilderName                     0.954462
LotSizeDimensions               0.936693
BuildingAreaTotal               0.933143
CoBuyerAgentFirstName           0.906543
ElementarySchool                0.869681
MiddleOrJuniorSchool            0.868618
HighSchool                      0.828579
CoListAgentFirstName            0.766505
CoListAgentLastName             0.765765
CoListOfficeName                0.751222
AssociationFeeFrequency         0.745475
SubdivisionName                 0.645693
Length: 23, dtype: float64
After feature selection: (1419

### Handling Invalid Numeric Values

In [6]:
df["NewConstructionYN"].unique().tolist()

[False, nan, True]

In [7]:
numeric_cols = df_current.select_dtypes(include="number").columns.tolist()
df_num = df_current[numeric_cols]
display(df_num.describe().T)

,count,mean,std,min,25%,50%,75%,max
ClosePrice,141990.0,1.344681e+06,8.084658e+06,1.750000,625000.000000,890000.000000,1.425000e+06,9.895000e+08
Latitude,141977.0,3.473271e+01,1.753996e+00,-22.863239,33.761396,34.083275,3.478326e+01,4.378444e+01
Longitude,141977.0,-1.185991e+02,3.191720e+00,-124.193201,-119.145860,-118.031107,-1.172602e+02,1.204327e+02
LivingArea,141912.0,2.048629e+03,1.044041e+03,0.000000,1386.000000,1820.000000,2.440000e+03,5.650000e+04
ParkingTotal,141989.0,3.058482e+00,4.408634e+01,-35.000000,2.000000,2.000000,3.000000e+00,1.572000e+04
YearBuilt,141893.0,1.975809e+03,2.760633e+01,1776.000000,1956.000000,1976.000000,1.998000e+03,2.026000e+03
BathroomsTotalInteger,141977.0,2.632455e+00,1.131931e+00,0.000000,2.000000,2.000000,3.000000e+00,3.500000e+01
BedroomsTotal,141990.0,3.492577e+00,9.635172e-01,0.000000,3.000000,3.000000,4.000000e+00,2.200000e+01
MainLevelBedrooms,86415.0,2.265590e+00,1.447396e+00,0.000000,1.000000,3.000000,3.000000e+00,4.400000e+01
GarageSpaces,136374.0,2.006524e+00,3.305410e+00,0.000000,2.000000,2.000000,2.000000e+00,6.000000e+02


In [8]:
df_current0 = df_current.copy()

def apply_invalid_rule(df, col, condition, rule_name):
    before_missing = df[col].isnull().sum()
    
    df.loc[condition, col] = np.nan
    
    after_missing = df[col].isnull().sum()

    
    print(f"{col}: {after_missing - before_missing} additional missing values after {rule_name}.")
    
    return df

# Invalid coordinates for CA
df_current0 = apply_invalid_rule(df_current0, "Latitude", (df_current0["Latitude"] < 32) | (df_current0["Latitude"] > 42), "Invalid Latitude cleaning")
df_current0 = apply_invalid_rule(df_current0, "Longitude", (df_current0["Longitude"] < -125) | (df_current0["Longitude"] > -114), "Invalid Longitude cleaning")
# Invalid numeric property feature
df_current0 = apply_invalid_rule(df_current0, "LotSizeSquareFeet", (df_current0["LotSizeSquareFeet"] <= 100), "Invalid LotSizeSquareFeet cleaning")
df_current0 = apply_invalid_rule(df_current0, "LivingArea", (df_current0["LivingArea"] <= 0), "Invalid LivingArea cleaning")
df_current0 = apply_invalid_rule(df_current0, "ParkingTotal", (df_current0["ParkingTotal"] < 0), "Invalid ParkingTotal cleaning")
df_current0 = apply_invalid_rule(df_current0, "GarageSpaces", (df_current0["GarageSpaces"] < 0), "Invalid GarageSpaces cleaning")
df_current0 = apply_invalid_rule(df_current0, "BathroomsTotalInteger", (df_current0["BathroomsTotalInteger"] <= 0), "Invalid BathroomsTotalInteger cleaning")
df_current0 = apply_invalid_rule(df_current0, "BedroomsTotal", (df_current0["BedroomsTotal"] <= 0), "Invalid BedroomsTotal cleaning")
df_current0 = apply_invalid_rule(df_current0, "MainLevelBedrooms", (df_current0["MainLevelBedrooms"] < 0), "Invalid MainLevelBedrooms cleaning")
df_current0 = apply_invalid_rule(df_current0,"AssociationFee",df_current0["AssociationFee"] < 0,"invalid association fee cleaning")
df_current0 = apply_invalid_rule(df_current0,"MainLevelBedrooms",df_current0["MainLevelBedrooms"] > df_current0["BedroomsTotal"],"main-level bedrooms consistency cleaning")

display(df_current0.describe())


Latitude: 24 additional missing values after Invalid Latitude cleaning.
Longitude: 36 additional missing values after Invalid Longitude cleaning.
LotSizeSquareFeet: 361 additional missing values after Invalid LotSizeSquareFeet cleaning.
LivingArea: 63 additional missing values after Invalid LivingArea cleaning.
ParkingTotal: 20 additional missing values after Invalid ParkingTotal cleaning.
GarageSpaces: 0 additional missing values after Invalid GarageSpaces cleaning.
BathroomsTotalInteger: 61 additional missing values after Invalid BathroomsTotalInteger cleaning.
BedroomsTotal: 80 additional missing values after Invalid BedroomsTotal cleaning.
MainLevelBedrooms: 0 additional missing values after Invalid MainLevelBedrooms cleaning.
AssociationFee: 0 additional missing values after invalid association fee cleaning.
MainLevelBedrooms: 834 additional missing values after main-level bedrooms consistency cleaning.


,CloseDate,ClosePrice,Latitude,Longitude,LivingArea,ParkingTotal,YearBuilt,BathroomsTotalInteger,BedroomsTotal,MainLevelBedrooms,GarageSpaces,AssociationFee,LotSizeSquareFeet
count,141990,1.419900e+05,141953.000000,141941.000000,141849.000000,141969.000000,141893.000000,141916.000000,141910.000000,85581.000000,136374.000000,100627.000000,1.391960e+05
mean,2025-11-10 22:11:32.769913,1.344681e+06,34.737919,-118.636990,2049.538551,3.059842,1975.808785,2.633586,3.494546,2.241607,2.006524,106.434293,3.720465e+05
min,2025-05-01 00:00:00,1.750000e+00,32.545249,-124.193201,100.000000,0.000000,1776.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.120000e+02
25%,2025-07-31 00:00:00,6.250000e+05,33.761541,-119.147099,1387.000000,2.000000,1956.000000,2.000000,3.000000,1.000000,2.000000,0.000000,5.663000e+03
50%,2025-11-03 00:00:00,8.900000e+05,34.083422,-118.031338,1820.000000,2.000000,1976.000000,2.000000,3.000000,3.000000,2.000000,0.000000,7.300000e+03
75%,2026-02-27 00:00:00,1.425000e+06,34.812377,-117.260348,2440.000000,3.000000,1998.000000,3.000000,4.000000,3.000000,2.000000,130.000000,1.045400e+04
max,2026-05-31 00:00:00,9.895000e+08,41.894714,-114.347957,56500.000000,15720.000000,2026.000000,35.000000,22.000000,16.000000,600.000000,20712.000000,1.938943e+09
std,NaN,8.084658e+06,1.696171,1.844576,1043.379482,44.089199,27.606333,1.130858,0.960213,1.359461,3.305410,345.323325,1.747105e+07


### Handling Count-Based Feature Sanity Checks

For count-based features, I used broad domain sanity checks instead of IQR-based thresholds. These variables are discrete counts and are usually concentrated around small integer values, so IQR can be too aggressive and may incorrectly flag valid larger homes as outliers.

`ParkingTotal` was given a higher upper bound because it may include driveway, open, or total available parking spaces. `GarageSpaces` specifically represents garage capacity, so it uses a stricter threshold.

Values above these broad upper bounds were converted to missing values rather than capped, because extremely large count values are more likely to be data quality issues than meaningful residential property characteristics.

This step is applied before the train/test split because the thresholds are manually defined domain rules, not thresholds learned from the data distribution.

In [9]:
"""
Apply rule-based sanity checks to count-based features.

These rules are manually defined domain checks, so they can be applied before
the train/test split. Values that are clearly unrealistic are converted to NaN
instead of removing the full row.
"""

df_current = df_current0.copy()

def apply_count_upper_rule(df, col, upper_bound):
    """
    Convert unrealistic count-based feature values to NaN using broad
    domain sanity checks. These are feature values, so rows are not removed.
    """
    before_missing = df[col].isnull().sum()
    rows_above = (df[col] > upper_bound).sum()
    max_before = df[col].max()
    
    df.loc[df[col] > upper_bound, col] = np.nan
    
    after_missing = df[col].isnull().sum()
    
    print(f"{col}:")
    print(f"  upper bound: {upper_bound}")
    print(f"  max before cleaning: {max_before}")
    print(f"  rows converted to NaN: {rows_above}")
    print(f"  additional missing values: {after_missing - before_missing}")
    
    return df


count_upper_rules = {
    "ParkingTotal": 50,
    "GarageSpaces": 20,
    "BedroomsTotal": 20,
    "BathroomsTotalInteger": 20,
    "MainLevelBedrooms": 20
}

for col, upper_bound in count_upper_rules.items():
    if col in df_current.columns:
        df_current = apply_count_upper_rule(df_current, col, upper_bound)

print("Shape after count-based sanity checks:", df_current.shape)

ParkingTotal:
  upper bound: 50
  max before cleaning: 15720.0
  rows converted to NaN: 73
  additional missing values: 73
GarageSpaces:
  upper bound: 20
  max before cleaning: 600.0
  rows converted to NaN: 19
  additional missing values: 19
BedroomsTotal:
  upper bound: 20
  max before cleaning: 22.0
  rows converted to NaN: 1
  additional missing values: 1
BathroomsTotalInteger:
  upper bound: 20
  max before cleaning: 35.0
  rows converted to NaN: 5
  additional missing values: 5
MainLevelBedrooms:
  upper bound: 20
  max before cleaning: 16.0
  rows converted to NaN: 0
  additional missing values: 0
Shape after count-based sanity checks: (141990, 22)


### Cleaning Categorical Features

In this step, I performed light cleaning on the remaining categorical features before train/test splitting. Extra spaces were stripped, empty strings were converted to missing values, and `PostalCode` was standardized as a 5-digit categorical location feature.

`Flooring` was handled separately because it contains multi-value material combinations. Instead of one-hot encoding the raw flooring combinations, I converted it into material indicator features such as carpet, tile, wood, vinyl, laminate, stone, concrete, bamboo, and brick. I also created a separate indicator for `SeeRemarks`, because it is not a flooring material but may indicate that flooring information is available only in listing remarks.

For `HighSchoolDistrict`, non-specific labels such as `Other`, `See Remarks`, `Call Listing Office`, and multi-district inquiry notes were treated as missing values because they do not identify a specific school district.

Boolean-like fields such as `ViewYN`, `PoolPrivateYN`, `AttachedGarageYN`, `FireplaceYN`, and `NewConstructionYN` already contain only `True`, `False`, and missing values. I kept them as categorical variables rather than converting them directly to 0/1, because missing values may represent unreported information rather than a true false value.

`Levels` was also handled as a multi-label categorical feature because it can contain combinations such as `One,Two`, `Two,One`, and `ThreeOrMore,MultiSplit`. Instead of one-hot encoding the raw combinations, I converted it into level indicator features for one-level, two-level, three-or-more-level, and multi-split structures. This prevents equivalent combinations with different ordering from being treated as separate categories.

Categorical imputation and one-hot encoding are not applied here. They will be fitted only on the training set after the train/test split to avoid data leakage.

In [10]:
"""
Clean categorical features before train/test split.

Only rule-based string cleaning and row-level feature extraction are applied here.
Imputation and encoding will be handled after the train/test split.
"""

categorical_cols = df_current.select_dtypes(exclude="number").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "CloseDate"]

# Strip spaces and convert empty strings to missing values
for col in categorical_cols:
    df_current[col] = df_current[col].astype("string").str.strip()
    df_current[col] = df_current[col].replace(r"^\s*$", np.nan, regex=True)


# PostalCode is a categorical location feature, not numeric.
# Keep only the first 5-digit ZIP code if ZIP+4 or mixed formats appear.
if "PostalCode" in df_current.columns:
    df_current["PostalCode"] = (
        df_current["PostalCode"]
        .astype("string")
        .str.extract(r"(\d{5})", expand=False)
    )


# Keep boolean-like categorical fields as True / False / missing.
# They are not converted to 0/1 because missing values may represent
# unreported information rather than a true False value.
yn_cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

for col in yn_cols:
    if col in df_current.columns:
        df_current[col] = df_current[col].astype("string").str.strip()

# Treat non-specific school district labels as missing.
# These values do not identify a specific district and are better handled
# as Unknown by the categorical imputer after the train/test split.
if "HighSchoolDistrict" in df_current.columns:
    ambiguous_school_pattern = (
        r"(?i)^(other|see remarks|call listing office|"
        r"more than 1 district.*|.*inquire.*)$"
    )
    
    df_current["HighSchoolDistrict"] = (
        df_current["HighSchoolDistrict"]
        .astype("string")
        .str.strip()
        .replace(r"^\s*$", np.nan, regex=True)
        .replace(ambiguous_school_pattern, np.nan, regex=True)
    )


# Convert Flooring from multi-value text into material indicator features.
# Use exact token matching because Flooring values are comma-separated material labels.
if "Flooring" in df_current.columns:
    flooring_clean = (
        df_current["Flooring"]
        .astype("string")
        .str.lower()
        .str.strip()
        .str.replace(r"\s*,\s*", ",", regex=True)
    )
    
    df_current["Flooring_missing"] = df_current["Flooring"].isna().astype(int)
    
    flooring_materials = {
        "has_carpet": "carpet",
        "has_tile": "tile",
        "has_wood": "wood",
        "has_vinyl": "vinyl",
        "has_laminate": "laminate",
        "has_stone": "stone",
        "has_concrete": "concrete",
        "has_bamboo": "bamboo",
        "has_brick": "brick"
    }
    
    for new_col, token in flooring_materials.items():
        pattern = rf"(?:^|,){re.escape(token)}(?:,|$)"
        df_current[new_col] = flooring_clean.str.contains(
            pattern,
            regex=True,
            na=False
        ).astype(int)
    
    # SeeRemarks is not a material, but it indicates that flooring details
    # may be described elsewhere in the listing.
    df_current["Flooring_see_remarks"] = flooring_clean.str.contains(
        r"(?:^|,)seeremarks(?:,|$)",
        regex=True,
        na=False
    ).astype(int)
    
    
    # Drop the raw multi-value text column to avoid high-cardinality one-hot encoding.
    df_current = df_current.drop(columns=["Flooring"], errors="ignore")

# Convert Levels from multi-value text into structure indicator features.
# Levels can contain ordered combinations such as "One,Two" and "Two,One",
# so raw one-hot encoding would incorrectly treat equivalent combinations
# as different categories.
if "Levels" in df_current.columns:
    levels_clean = (
        df_current["Levels"]
        .astype("string")
        .str.lower()
        .str.strip()
        .str.replace(r"\s*,\s*", ",", regex=True)
    )
    
    df_current["Levels_missing"] = df_current["Levels"].isna().astype(int)
    
    level_patterns = {
        "level_one": "one",
        "level_two": "two",
        "level_three_or_more": "threeormore",
        "level_multisplit": "multisplit"
    }
    
    for new_col, token in level_patterns.items():
        pattern = rf"(?:^|,){re.escape(token)}(?:,|$)"
        df_current[new_col] = levels_clean.str.contains(
            pattern,
            regex=True,
            na=False
        ).astype(int)
    
    # Drop the raw multi-value Levels column to avoid high-cardinality
    # one-hot encoding of ordered combinations.
    df_current = df_current.drop(columns=["Levels"], errors="ignore")
    


print("Categorical cleaning completed.")
print("Shape after categorical cleaning:", df_current.shape)

Categorical cleaning completed.
Shape after categorical cleaning: (141990, 36)


In [11]:
"""
Check categorical columns after basic categorical cleaning.
"""

categorical_cols = df_current.select_dtypes(exclude="number").columns.tolist()
categorical_cols = [col for col in categorical_cols if col != "CloseDate"]

cat_summary = []

for col in categorical_cols:
    cat_summary.append({
        "column": col,
        "missing_count": df_current[col].isnull().sum(),
        "missing_rate": df_current[col].isnull().mean(),
        "unique_count": df_current[col].nunique(dropna=True),
        "top_value": df_current[col].mode(dropna=True).iloc[0] if df_current[col].notna().any() else np.nan,
        "top_value_count": df_current[col].value_counts(dropna=True).iloc[0] if df_current[col].notna().any() else 0
    })

cat_summary = pd.DataFrame(cat_summary).sort_values("unique_count", ascending=False)
display(cat_summary)

,column,missing_count,missing_rate,unique_count,top_value,top_value_count
6,PostalCode,2,0.000014,1439,92253,989
5,HighSchoolDistrict,47125,0.331890,422,Los Angeles Unified,8461
0,ViewYN,12901,0.090859,2,True,79792
2,AttachedGarageYN,17193,0.121086,2,True,104119
1,PoolPrivateYN,10978,0.077315,2,False,108956
4,NewConstructionYN,10829,0.076266,2,False,126269
3,FireplaceYN,108,0.000761,2,True,103390


### Time-Based Train/Test Split

I used a time-based train/test split because the goal is to predict property close prices in a realistic future-data setting. The most recent available sale month is used as the test set, and all earlier months are used as the training set.

This split is placed before target outlier filtering, continuous feature capping, imputation, and encoding because those steps learn thresholds or mappings from the data and should be fitted only on the training set.

In [12]:
"""
Create a time-based train/test split.

The most recent available CloseDate month is used as the test set.
All earlier months are used as the training set.
"""

df_model_base = df_current.copy()

df_model_base["CloseDate"] = pd.to_datetime(df_model_base["CloseDate"], errors="coerce")
df_model_base["SaleMonth"] = df_model_base["CloseDate"].dt.to_period("M")

test_month = df_model_base["SaleMonth"].max()

train_df = df_model_base[df_model_base["SaleMonth"] < test_month].copy()
test_df = df_model_base[df_model_base["SaleMonth"] == test_month].copy()

print("Test month:", test_month)
print("Train shape before target outlier handling:", train_df.shape)
print("Test shape:", test_df.shape)

Test month: 2026-05
Train shape before target outlier handling: (129966, 37)
Test shape: (12024, 37)


### Handling Target Outliers in the Training Set

`ClosePrice` is the prediction target, so clearly abnormal target values should not be imputed. To avoid using information from the test set, target outlier thresholds are calculated using only the training data.

I created `PricePerLivingArea` only for target outlier inspection. This metric helps evaluate whether a sale price is reasonable relative to the property's living area. It is not used as a modeling feature because it is derived from the target variable and would cause target leakage.

The test set is left unchanged so that model evaluation remains closer to a realistic future-data setting.

In [13]:
"""
Handle target outliers in the training set only.

The test set is not filtered here because it is used as the final evaluation set.
"""

train_df = train_df.copy()

# Create price per living area for target outlier inspection only
train_df["PricePerLivingArea"] = (
    train_df["ClosePrice"] / train_df["LivingArea"]
).replace([np.inf, -np.inf], np.nan)

print("Describe PricePerLivingArea for extreme low ClosePrice observations in training data:")
print(
    train_df.loc[
        train_df["ClosePrice"] < train_df["ClosePrice"].quantile(0.001),
        "PricePerLivingArea"
    ].describe()
)

print("---------")

print("Describe PricePerLivingArea for extreme high ClosePrice observations in training data:")
print(
    train_df.loc[
        train_df["ClosePrice"] > train_df["ClosePrice"].quantile(0.999),
        "PricePerLivingArea"
    ].describe()
)

# Low-end target outliers
low_price_threshold = train_df["ClosePrice"].quantile(0.001)
extreme_low_mask = train_df["ClosePrice"] < low_price_threshold

# High-end target outliers based on price per living area
high_price_threshold = train_df["ClosePrice"].quantile(0.999)
high_price_rows = train_df[train_df["ClosePrice"] > high_price_threshold].copy()

q1 = high_price_rows["PricePerLivingArea"].quantile(0.25)
q3 = high_price_rows["PricePerLivingArea"].quantile(0.75)
iqr = q3 - q1

if pd.notna(iqr) and iqr > 0:
    price_per_area_cutoff = q3 + 1.5 * iqr
else:
    price_per_area_cutoff = np.inf

extreme_high_mask = train_df["PricePerLivingArea"] > price_per_area_cutoff

print("Low ClosePrice threshold:", low_price_threshold)
print("Rows removed from train low end:", extreme_low_mask.sum())
print("Price per living area upper cutoff:", price_per_area_cutoff)
print("Rows removed from train high end:", extreme_high_mask.sum())

# Remove target outliers from training data only
train_df = train_df[
    (~extreme_low_mask) & (~extreme_high_mask)
].copy()

# Drop inspection-only variable to avoid target leakage
train_df = train_df.drop(columns=["PricePerLivingArea"], errors="ignore")

print("Train shape after target outlier handling:", train_df.shape)
print("Test shape unchanged:", test_df.shape)

Describe PricePerLivingArea for extreme low ClosePrice observations in training data:
count    130.000000
mean      71.783290
std       55.134116
min        0.000498
25%       35.370050
50%       60.512127
75%       88.186553
max      338.541667
Name: PricePerLivingArea, Length: 8, dtype: float64
---------
Describe PricePerLivingArea for extreme high ClosePrice observations in training data:
count    1.220000e+02
mean     7.115185e+04
std      1.964029e+05
min      9.012489e+02
25%      2.185185e+03
50%      2.781201e+03
75%      4.989449e+03
max      1.164067e+06
Name: PricePerLivingArea, Length: 8, dtype: float64
Low ClosePrice threshold: 92482.5
Rows removed from train low end: 130
Price per living area upper cutoff: 9195.845450139343
Rows removed from train high end: 23
Train shape after target outlier handling: (129813, 37)
Test shape unchanged: (12024, 37)


### Splitting Features and Target

After the time-based split and target outlier handling on the training set, I separated the target variable from the feature matrix.

`ClosePrice` is used as the target. `CloseDate` and `SaleMonth` are removed from the feature matrix because they are used for splitting, not as baseline model predictors.

In [14]:
"""
Separate features and target after train/test split.
"""

y_train = train_df["ClosePrice"]
y_test = test_df["ClosePrice"]

X_train = train_df.drop(columns=["ClosePrice", "CloseDate", "SaleMonth"], errors="ignore")
X_test = test_df.drop(columns=["ClosePrice", "CloseDate", "SaleMonth"], errors="ignore")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (129813, 34)
X_test shape: (12024, 34)
y_train shape: (129813,)
y_test shape: (12024,)


### Train-Only Continuous Feature Capping

For highly skewed continuous features, I capped values above the 99.9th percentile. The cap thresholds are calculated using only the training set, then the same thresholds are applied to both the training and test sets.

This avoids data leakage because the test set is not used to determine preprocessing thresholds.

In [15]:
"""
Fit continuous feature caps on X_train only, then apply the same caps to X_train and X_test.
"""

continuous_cap_cols = [
    "LivingArea",
    "LotSizeSquareFeet",
    "AssociationFee"
]

continuous_cap_cols = [col for col in continuous_cap_cols if col in X_train.columns]

cap_values = {}

for col in continuous_cap_cols:
    cap_values[col] = X_train[col].quantile(0.999)

def apply_continuous_caps(X, cap_values):
    """
    Apply pre-computed upper cap values to continuous features.
    The cap values should be fitted on the training set only.
    """
    X = X.copy()
    
    for col, cap in cap_values.items():
        if col in X.columns and pd.notna(cap):
            X[col] = X[col].clip(upper=cap)
    
    return X

X_train = apply_continuous_caps(X_train, cap_values)
X_test = apply_continuous_caps(X_test, cap_values)

cap_summary = pd.DataFrame({
    "column": list(cap_values.keys()),
    "train_q999_cap": list(cap_values.values()),
    "train_max_after_cap": [X_train[col].max() for col in cap_values.keys()],
    "test_max_after_cap": [X_test[col].max() for col in cap_values.keys()]
})

display(cap_summary)

,column,train_q999_cap,train_max_after_cap,test_max_after_cap
0,LivingArea,1.027060e+04,1.027060e+04,1.027060e+04
1,LotSizeSquareFeet,5.959876e+06,5.959876e+06,5.959876e+06
2,AssociationFee,4.187875e+03,4.187875e+03,4.187875e+03


### Fitted preprocessing pipeline for model input

This cell creates a train-fitted preprocessing pipeline for converting the cleaned feature tables into model-ready matrices.

To prevent data leakage, all preprocessing parameters are learned from `X_train` only and then applied to `X_test`. Numeric missing values are handled through a configurable imputation strategy. The median option provides a robust and interpretable baseline, while the KNN option estimates missing values using distance-based similarity after standardizing numeric features. Both options preserve missingness information by adding missing indicators.

Categorical features are imputed as `"Unknown"` and one-hot encoded. Rare categories are grouped with `min_frequency=50` to reduce sparsity and overfitting, while unseen categories in the test set are handled safely.

The resulting `X_train_processed` and `X_test_processed` matrices are used for downstream modeling, while the original cleaned feature tables remain readable for CSV export.

In [16]:
"""
Build preprocessing pipeline with switchable numeric imputation method.

Options:
- median: median imputation + missing indicators
- knn: KNN imputation + missing indicators

The preprocessor is fitted on X_train only and then used to transform both
X_train and X_test.
"""

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# -----------------------------
# Config
# -----------------------------
imputer_method = "knn"   # options: "median", "knn"
knn_neighbors = 5
scale_numeric = True        # Recommended if using linear models


# -----------------------------
# Make copies before preprocessing
# -----------------------------
X_train = X_train.copy()
X_test = X_test.copy()

# Safety: make sure target/date/leakage columns are not in X
drop_from_X = ["ClosePrice", "CloseDate", "SaleMonth", "PricePerLivingArea"]

X_train = X_train.drop(columns=drop_from_X, errors="ignore")
X_test = X_test.drop(columns=drop_from_X, errors="ignore")


# -----------------------------
# Convert pandas nullable missing values to np.nan
# -----------------------------
string_cols = X_train.select_dtypes(include=["string"]).columns.tolist()

for col in string_cols:
    X_train[col] = X_train[col].astype(object)
    X_test[col] = X_test[col].astype(object)

X_train = X_train.where(pd.notna(X_train), np.nan)
X_test = X_test.where(pd.notna(X_test), np.nan)


# -----------------------------
# Identify numeric and categorical columns
# -----------------------------
numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()



# -----------------------------
# Numeric preprocessing
# -----------------------------
if imputer_method == "median":
    numeric_steps = [
        ("imputer", SimpleImputer(
            strategy="median",
            add_indicator=True
        ))
    ]

    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)


elif imputer_method == "knn":
    # KNN uses distance, so numeric features should be scaled before imputation.
    numeric_transformer = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("imputer", KNNImputer(
            n_neighbors=knn_neighbors,
            weights="distance",
            add_indicator=True
        ))
    ])


else:
    raise ValueError("imputer_method must be either 'median' or 'knn'")


# -----------------------------
# Categorical preprocessing
# -----------------------------
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        min_frequency=50
    ))
])


# -----------------------------
# Combine numeric and categorical preprocessing
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)


# -----------------------------
# Fit on train, transform train and test
# -----------------------------
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


# -----------------------------
# Check results
# -----------------------------
print("Imputer method:", imputer_method)
print("KNN neighbors:", knn_neighbors if imputer_method == "knn" else None)
print("Scale numeric:", scale_numeric)
print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Imputer method: knn
KNN neighbors: 5
Scale numeric: True
Numeric columns: ['Latitude', 'Longitude', 'LivingArea', 'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'BedroomsTotal', 'MainLevelBedrooms', 'GarageSpaces', 'AssociationFee', 'LotSizeSquareFeet', 'Flooring_missing', 'has_carpet', 'has_tile', 'has_wood', 'has_vinyl', 'has_laminate', 'has_stone', 'has_concrete', 'has_bamboo', 'has_brick', 'Flooring_see_remarks', 'Levels_missing', 'level_one', 'level_two', 'level_three_or_more', 'level_multisplit']
Categorical columns: ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN', 'HighSchoolDistrict', 'PostalCode']
Processed train shape: (129813, 967)
Processed test shape: (12024, 967)


### Save KNN-imputed datasets before and after encoding

This cell saves two KNN-imputed datasets for team use.

`model_data_knn_before_encoding.csv` is the readable cleaned modeling dataset. Numeric features are KNN-imputed and converted back to their original scale. Categorical missing values are filled as `Unknown`, but categorical columns are not one-hot encoded.

`model_data_knn_after_encoding.csv.gz` is the model-ready encoded dataset. It uses the fitted preprocessing pipeline output, so numeric features are KNN-imputed and scaled, while categorical features are imputed and one-hot encoded. This file is compressed because the encoded feature matrix is high-dimensional.

Both files include `split`, `CloseDate`, `SaleMonth`, and `ClosePrice` so the team can separate train/test rows and access the target variable directly.

In [17]:
"""
Save two KNN-imputed modeling CSV files:

1. model_data_knn_before_encoding.csv
   - readable cleaned table
   - numeric features are KNN-imputed and converted back to original scale
   - categorical missing values are filled as Unknown
   - categorical columns are not one-hot encoded

2. model_data_knn_after_encoding.csv.gz
   - model-ready encoded table
   - numeric features are KNN-imputed and scaled
   - categorical features are imputed and one-hot encoded
   - compressed CSV to reduce file size

This cell does not refit KNN. It reuses the already fitted preprocessor and
the already generated X_train_processed / X_test_processed matrices.
"""

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from scipy import sparse

output_dir = Path.cwd()

if imputer_method != "knn":
    raise ValueError("This export cell is intended for KNN-imputed outputs. Set imputer_method = 'knn' first.")


# -----------------------------
# Helper function
# -----------------------------
def to_dense_array(matrix_slice):
    """Convert a sparse or dense matrix slice to a NumPy array."""
    if sparse.issparse(matrix_slice):
        return matrix_slice.toarray()
    return np.asarray(matrix_slice)


def count_nan_in_matrix(matrix):
    """Efficiently count NaN values in dense or sparse matrices."""
    if sparse.issparse(matrix):
        return np.isnan(matrix.data).sum()
    return np.isnan(matrix).sum()


# -----------------------------
# 1. Build before-encoding readable dataset
# -----------------------------
X_train_before = X_train.copy()
X_test_before = X_test.copy()

drop_from_X = ["ClosePrice", "CloseDate", "SaleMonth", "PricePerLivingArea"]

X_train_before = X_train_before.drop(columns=drop_from_X, errors="ignore")
X_test_before = X_test_before.drop(columns=drop_from_X, errors="ignore")

# Convert pandas nullable string columns to object dtype
string_cols = X_train_before.select_dtypes(include=["string"]).columns.tolist()

for col in string_cols:
    X_train_before[col] = X_train_before[col].astype(object)
    X_test_before[col] = X_test_before[col].astype(object)

# Convert pandas missing values, including pd.NA, to np.nan
X_train_before = X_train_before.where(pd.notna(X_train_before), np.nan)
X_test_before = X_test_before.where(pd.notna(X_test_before), np.nan)

base_feature_cols = X_train_before.columns.tolist()


# -----------------------------
# 2. Add readable missing indicator columns
# -----------------------------
# Use training set only to decide which columns get missing indicators.
missing_indicator_source_cols = [
    col for col in base_feature_cols
    if X_train_before[col].isna().any()
]

missing_indicator_cols = []

for col in missing_indicator_source_cols:
    indicator_col = f"{col}_missing_ind"

    X_train_before[indicator_col] = X_train_before[col].isna().astype(int)
    X_test_before[indicator_col] = X_test_before[col].isna().astype(int)

    missing_indicator_cols.append(indicator_col)

print("Missing indicator columns added:", missing_indicator_cols)


# -----------------------------
# 3. Fill numeric values using already processed KNN output
# -----------------------------
# The first len(numeric_cols) columns of X_train_processed / X_test_processed
# are the KNN-imputed numeric values in scaled space.
# Convert them back to original scale for the before-encoding CSV.

n_num = len(numeric_cols)

X_train_num_scaled = to_dense_array(X_train_processed[:, :n_num])
X_test_num_scaled = to_dense_array(X_test_processed[:, :n_num])

scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]

X_train_before[numeric_cols] = scaler.inverse_transform(X_train_num_scaled)
X_test_before[numeric_cols] = scaler.inverse_transform(X_test_num_scaled)


# -----------------------------
# 4. Fill categorical values using fitted categorical imputer
# -----------------------------
cat_pipeline = preprocessor.named_transformers_["cat"]
categorical_imputer = cat_pipeline.named_steps["imputer"]

if categorical_cols:
    X_train_before[categorical_cols] = categorical_imputer.transform(
        X_train_before[categorical_cols]
    )

    X_test_before[categorical_cols] = categorical_imputer.transform(
        X_test_before[categorical_cols]
    )


# -----------------------------
# 5. Add split, CloseDate, SaleMonth, and ClosePrice back
# -----------------------------
train_before_output = X_train_before.copy()
test_before_output = X_test_before.copy()

train_before_output.insert(0, "split", "train")
test_before_output.insert(0, "split", "test")

train_before_output.insert(
    1,
    "CloseDate",
    train_df.loc[X_train.index, "CloseDate"].values
)

test_before_output.insert(
    1,
    "CloseDate",
    test_df.loc[X_test.index, "CloseDate"].values
)

train_before_output.insert(
    2,
    "SaleMonth",
    train_df.loc[X_train.index, "SaleMonth"].astype(str).values
)

test_before_output.insert(
    2,
    "SaleMonth",
    test_df.loc[X_test.index, "SaleMonth"].astype(str).values
)

train_before_output["ClosePrice"] = y_train.loc[X_train.index].values
test_before_output["ClosePrice"] = y_test.loc[X_test.index].values

model_data_knn_before_encoding = pd.concat(
    [train_before_output, test_before_output],
    axis=0,
    ignore_index=True
)


# -----------------------------
# 6. Save before-encoding CSV
# -----------------------------
before_encoding_path = output_dir / "model_data_knn_before_encoding.csv"

model_data_knn_before_encoding.to_csv(before_encoding_path, index=False)

print(f"Saved before-encoding CSV to: {before_encoding_path}")
print("Before-encoding shape:", model_data_knn_before_encoding.shape)
print("Before-encoding missing values:", model_data_knn_before_encoding.isna().sum().sum())


# -----------------------------
# 7. Build after-encoding model-ready dataset
# -----------------------------
try:
    encoded_feature_names = preprocessor.get_feature_names_out()
except Exception:
    encoded_feature_names = [
        f"encoded_feature_{i}"
        for i in range(X_train_processed.shape[1])
    ]


def processed_matrix_to_dataframe(matrix, columns):
    """
    Convert sklearn processed matrix to a pandas DataFrame.

    Uses a pandas sparse DataFrame when the matrix is sparse to reduce memory use.
    """
    if sparse.issparse(matrix):
        return pd.DataFrame.sparse.from_spmatrix(
            matrix,
            columns=columns
        )
    else:
        return pd.DataFrame(
            matrix,
            columns=columns
        )


X_train_encoded_df = processed_matrix_to_dataframe(
    X_train_processed,
    encoded_feature_names
)

X_test_encoded_df = processed_matrix_to_dataframe(
    X_test_processed,
    encoded_feature_names
)


# -----------------------------
# 8. Add metadata and target to encoded dataset
# -----------------------------
train_encoded_meta = pd.DataFrame({
    "split": "train",
    "CloseDate": train_df.loc[X_train.index, "CloseDate"].values,
    "SaleMonth": train_df.loc[X_train.index, "SaleMonth"].astype(str).values,
    "ClosePrice": y_train.loc[X_train.index].values
})

test_encoded_meta = pd.DataFrame({
    "split": "test",
    "CloseDate": test_df.loc[X_test.index, "CloseDate"].values,
    "SaleMonth": test_df.loc[X_test.index, "SaleMonth"].astype(str).values,
    "ClosePrice": y_test.loc[X_test.index].values
})

train_after_output = pd.concat(
    [
        train_encoded_meta.reset_index(drop=True),
        X_train_encoded_df.reset_index(drop=True)
    ],
    axis=1
)

test_after_output = pd.concat(
    [
        test_encoded_meta.reset_index(drop=True),
        X_test_encoded_df.reset_index(drop=True)
    ],
    axis=1
)

model_data_knn_after_encoding = pd.concat(
    [train_after_output, test_after_output],
    axis=0,
    ignore_index=True
)


# -----------------------------
# 9. Save after-encoding CSV
# -----------------------------
# Use gzip compression because the encoded matrix is much wider.
after_encoding_path = output_dir / "model_data_knn_after_encoding.csv.gz"

model_data_knn_after_encoding.to_csv(
    after_encoding_path,
    index=False,
    compression="gzip"
)

print(f"Saved after-encoding CSV to: {after_encoding_path}")
print("After-encoding shape:", model_data_knn_after_encoding.shape)
print(
    "Encoded matrix missing values:",
    count_nan_in_matrix(X_train_processed) + count_nan_in_matrix(X_test_processed)
)


# -----------------------------
# 10. Save fitted preprocessing artifacts
# -----------------------------
preprocessing_artifacts = {
    "cap_values": cap_values,
    "preprocessor": preprocessor,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "feature_columns": X_train.columns.tolist(),
    "base_feature_cols_before_encoding": base_feature_cols,
    "missing_indicator_cols_before_encoding": missing_indicator_cols,
    "target": "ClosePrice",
    "test_month": str(test_month),
    "model_preprocessor_imputer_method": imputer_method,
    "knn_neighbors": knn_neighbors,
    "scale_numeric": scale_numeric,
    "before_encoding_csv": before_encoding_path.name,
    "after_encoding_csv": after_encoding_path.name
}

artifact_path = output_dir / "preprocessing_artifacts.joblib"

joblib.dump(preprocessing_artifacts, artifact_path)

print(f"Saved preprocessing artifacts to: {artifact_path}")

Missing indicator columns added: ['ViewYN_missing_ind', 'PoolPrivateYN_missing_ind', 'Latitude_missing_ind', 'Longitude_missing_ind', 'LivingArea_missing_ind', 'AttachedGarageYN_missing_ind', 'ParkingTotal_missing_ind', 'YearBuilt_missing_ind', 'BathroomsTotalInteger_missing_ind', 'BedroomsTotal_missing_ind', 'FireplaceYN_missing_ind', 'MainLevelBedrooms_missing_ind', 'NewConstructionYN_missing_ind', 'GarageSpaces_missing_ind', 'HighSchoolDistrict_missing_ind', 'PostalCode_missing_ind', 'AssociationFee_missing_ind', 'LotSizeSquareFeet_missing_ind']
Saved before-encoding CSV to: d:\IDX_Exchange\IDX-Exchange-summer-2026\Week3\model_data_knn_before_encoding.csv
Before-encoding shape: (141837, 56)
Before-encoding missing values: 0
Saved after-encoding CSV to: d:\IDX_Exchange\IDX-Exchange-summer-2026\Week3\model_data_knn_after_encoding.csv.gz
After-encoding shape: (141837, 971)
Encoded matrix missing values: 0
Saved preprocessing artifacts to: d:\IDX_Exchange\IDX-Exchange-summer-2026\Week3\

In [20]:
df_after = pd.read_csv("model_data_knn_after_encoding.csv.gz")

[c for c in df_after.columns if "infrequent" in c.lower()]

['cat__HighSchoolDistrict_infrequent_sklearn',
 'cat__PostalCode_infrequent_sklearn']